# Les importations

In [38]:
%pip install evaluate
%pip install sacrebleu rouge_score
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
%pip install typing_extensions==4.15.0 torch==2.7.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126




Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu126
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 170.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 238.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 189.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 72.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 85.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 115.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 158.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 12

In [20]:
# Les importations
from transformers import get_linear_schedule_with_warmup, AutoTokenizer,TrainerCallback,AutoModelForCausalLM, DataCollatorForSeq2Seq,AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer,EarlyStoppingCallback
from datasets import load_dataset,Dataset, DatasetDict
import evaluate
import numpy as np
from torch.cuda.amp import autocast, GradScaler
import torch
import tensorflow as tf
import matplotlib.pyplot as plt

In [21]:
# notre fichier d'arabe
#with open("/kaggle/input/arb-fr-2/arabe_tchad3.txt", "r") as file:
with open("/kaggle/input/arb-fr/for_dataset_arb.txt", "r") as file:
    # On lit le contenu du fichier
    data_arb = file.readlines()
# notre fichier de français
#with open("/kaggle/input/arb-fr-2/V_francais_3.txt", "r") as file:
with open("/kaggle/input/arb-fr/for_dataset_fr.txt", "r") as file:
    # On lit le contenu du fichier
    data_fr = file.readlines()

# Supprimons les lignes vides
data_fr = [line for line in data_fr if line.strip()]
data_arb = [line for line in data_arb if line.strip()]
print("fr_lengh :", len(data_fr))
print("arb_lengh :", len(data_arb))


fr_lengh : 13003
arb_lengh : 13003


In [22]:
data_arb[:10]

['Fi l‑bidaaya,Allah khalag al‑samaawaat wa l‑ard.\n',
 'Wa l‑ard faadiye.Hi ma indaha cheyy foogha.Wa l‑dalaam gaaʼid yikhatti al‑almi al‑khariig wa Ruuh Allah gaaʼid yuhuum fi l‑almi.\n',
 'Wa Allah al‑Rabb gaal\u202f:\u202fKhalli yukuun nuur. Wa khalaas,al‑nuur kaan.\n',
 'Wa Allah al‑Rabb chaaf kadar al‑nuur da sameh.Wa farag al‑nuur min al‑dalaam.\n',
 'Wa samma al‑nuur nahaar wa l‑dalaam leel.Wa kaan achiiye wa kaan fajur.Da l‑yoom al‑awwal.\n',
 'Wa Allah al‑Rabb gaal\u202f:\u202fKhalli tukuun faraga been al‑almi wa tifaarig al‑almi min al‑almi.\u202f\n',
 'Wa l‑Rabb khalag al‑faraga wa farag beeha al‑almi al‑tihit min al‑almi al‑foog.Wa khalaas,bigi misil da.\n',
 'Wa samma al‑faraga di,sama.Wa kaan achiiye wa kaan fajur.Da l‑yoom al‑taani.\n',
 'Wa l‑Rabb gaal\u202f:\u202fKhalli al‑almi al‑tihit le l‑sama yilimm bakaan waahid wa yikhalli bakaan yaabis yibiin. Wa khalaas,bigi misil da.\n',
 'Wa hu samma al‑bakaan al‑yaabis,ard wa l‑bakaan al‑indah almi,bahar.Wa Allah al‑Rabb ch

In [23]:
data_fr[:10]

['Au commencement, Dieu créa les cieux et la terre.\n',
 "La terre était informe et vide: il y avait des ténèbres à la surface de l'abîme, et l'esprit de Dieu se mouvait au-dessus des eaux.\n",
 'Dieu dit: Que la lumière soit! Et la lumière fut.\n',
 "Dieu vit que la lumière était bonne; et Dieu sépara la lumière d'avec les ténèbres.\n",
 'Dieu appela la lumière jour, et il appela les ténèbres nuit. Ainsi, il y eut un soir, et il y eut un matin: ce fut le premier jour.\n',
 "Dieu dit: Qu'il y ait une étendue entre les eaux, et qu'elle sépare les eaux d'avec les eaux.\n",
 "Et Dieu fit l'étendue, et il sépara les eaux qui sont au-dessous de l'étendue d'avec les eaux qui sont au-dessus de l'étendue. Et cela fut ainsi.\n",
 "Dieu appela l'étendue ciel. Ainsi, il y eut un soir, et il y eut un matin: ce fut le second jour.\n",
 'Dieu dit: Que les eaux qui sont au-dessous du ciel se rassemblent en un seul lieu, et que le sec paraisse. Et cela fut ainsi.\n',
 "Dieu appela le sec terre, et il 

In [24]:
# Création du dictionnaire
dataset_dict = {"translation": [{"arb": a, "fr": f} for a, f in zip(data_arb, data_fr)]}

# Création du dataset
dataset = Dataset.from_dict(dataset_dict)
# On va diviser notre dataset en deux parties
 #train_dataset =  DatasetDict({"train":test['train']})

train_test_dataset = dataset.train_test_split(test_size=0.2, seed= 42, shuffle= True)

test_val_dataset = train_test_dataset['test'].train_test_split(test_size= 0.5, seed= 42)
train_dataset = DatasetDict ({
    'train' : train_test_dataset['train'],
    'val'   : test_val_dataset['train'],
    'test'  : test_val_dataset['test']
})

In [25]:
train_dataset

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 10402
    })
    val: Dataset({
        features: ['translation'],
        num_rows: 1300
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 1301
    })
})

### Chargement de notre dataset

In [26]:
def extract_langage(examples) :
    # On va extraire le langage arabe et le langage français
    inputs = [arb['arb'] for arb in examples["translation"]]
    target = [fr['fr'] for fr in examples["translation"]]
    return {'inputs' : inputs, 'target' : target}

# On va appliquer la fonction sur notre dataset
train_dataset = train_dataset.map(extract_langage, batched=True, remove_columns=["translation"])
print(train_dataset['train']["inputs"][0])
print(train_dataset["train"]["target"][0])


Map:   0%|          | 0/10402 [00:00<?, ? examples/s]

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1301 [00:00<?, ? examples/s]

Wa yaatu al‑karab raas bineeye wa lissaaʼ ma chaalha ?Al‑naadum da,khalli yigabbil beetah achaan akuun hu yumuut fi l‑harib wa naadum aakhar yichiilha. 

Qui est-ce qui a fiancé une femme, et ne l'a point encore prise? Qu'il s'en aille et retourne chez lui, de peur qu'il ne meure dans la bataille et qu'un autre ne la prenne.



In [27]:
print(train_dataset['val']["inputs"][0])
print(train_dataset["val"]["target"][0])

Wa inta tihajji leyah be l‑cheyy al‑waajib yuguulah.Wa ana zaati nukuun maʼaak inta wa maʼa Haaruun.Wa niʼooriiku be l‑cheyy al‑waajib tisawwuuh.

Tu lui parleras, et tu mettras les paroles dans sa bouche; et moi, je serai avec ta bouche et avec sa bouche, et je vous enseignerai ce que vous aurez à faire.



### Pretraitement de notre dataset

In [28]:
from huggingface_hub import notebook_login

notebook_login()

### Evaluation du model

In [29]:
# Charger les métriques
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")
#accuracy_metric = evaluate.load("accuracy")  # Pour évaluer la précision


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [30]:
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]  # ROUGE/METEOR attend des listes imbriquées
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Décodage des prédictions et labels
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Nettoyage du texte
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    # 🏆 Calcul des métriques
    bleu_result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    rouge_result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
    meteor_result = meteor_metric.compute(predictions=decoded_preds, references=decoded_labels)
    # accuracy_result = accuracy_metric.compute(predictions=decoded_preds, references=decoded_labels) # Removed accuracy metric

    # 📊 Récupération des scores
    result = {
        "bleu": bleu_result["score"],
        "rouge": rouge_result["rougeL"],  # Utilisation du score ROUGE-L
        "meteor": meteor_result["meteor"],
        # "accuracy": accuracy_result["accuracy"]  # Précision des prédictions # Removed accuracy metric
    }

    # Calcul de la longueur moyenne des prédictions
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)

    # Arrondir les scores
    result = {k: round(v, 4) for k, v in result.items()}
    return result

In [31]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [32]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [33]:
early =  EarlyStoppingCallback(early_stopping_patience = 3)
#early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

class MemoryCleanupCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()  # Libérer la mémoire GPU inutilisée


class AMPCallback(TrainerCallback):
    def __init__(self):  #torch.amp.GradScaler('cuda', args...)
        super().__init__()
        self.scaler = GradScaler('cuda')  # Initialise le scaler pour AMP

    def on_step_begin(self, args, state, control, **kwargs):
        control.should_log = True  # Forcer l'affichage des logs

    def on_backward_end(self, args, state, control, **kwargs):
        with autocast():  # Active AMP pour les calculs
            self.scaler.scale(state.loss).backward()
            self.scaler.step(state.optimizer)
            self.scaler.update()


# Le modele Facebook multilingue

In [16]:
checkpoint = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint, token = os.environ.get('TOKEN'))

# Appliquons la tokenisation sur notre dataset
max_input_length = 128
max_target_length = 128

# Dans votre code pour Arabe Tchadien (Latin) -> Français
SOURCE_LANG_CODE_NLLB = "arb_Latn"
TARGET_LANG_CODE_NLLB = "fra_Latn"

def preprocess_function(examples):
    tokenizer.src_lang = SOURCE_LANG_CODE_NLLB
    model_inputs = tokenizer(
        examples["inputs"],
        max_length=max_input_length,
        truncation=True
    )

    # Nettoyer les cibles : enlever None ou chaînes vides
    clean_targets = []
    for t in examples["target"]:
        if isinstance(t, str) and t.strip():
            clean_targets.append(t)
        else:
            clean_targets.append(" ")

    tokenizer.tgt_lang = TARGET_LANG_CODE_NLLB
    labels = tokenizer(
        clean_targets,
        max_length=max_target_length,
        truncation=True,
        text_target=clean_targets  # Pour compatibilité future
    )

    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs
# On va appliquer la fonction sur notre dataset
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=["inputs", "target"])

# On va maintenant definir notre collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Map:   0%|          | 0/10402 [00:00<?, ? examples/s]

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1301 [00:00<?, ? examples/s]

In [17]:
from transformers import TrainerCallback

class LossTrackerCallback(TrainerCallback):
    def __init__(self):
        self.history = {
            "step": [],
            "train_loss": [],
            "eval_loss": []
        }

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        
        if "loss" in logs:
            self.history["step"].append(state.global_step)
            self.history["train_loss"].append(logs["loss"])
            self.history["eval_loss"].append(None)

        if "eval_loss" in logs:
            self.history["step"].append(state.global_step)
            self.history["train_loss"].append(None)
            self.history["eval_loss"].append(logs["eval_loss"])


In [18]:

training_args = Seq2SeqTrainingArguments(
    #output_dir="/kaggle/tmp/my_train_model_facebook_nllb_200",
    #run_name="nllb_experiment_v1",   Nom de run distinct
    output_dir="/kaggle/working/my_train_model_facebook_nllb_200",
    report_to="none",
    hub_model_id=os.environ.get('HUB_ID_Fr'),  # ✅ Ajoute ton Repo ID
    eval_strategy="steps",
    learning_rate=2e-5,
    #learning_rate=1e-5,
    gradient_checkpointing=True,  # ✅ Active la réduction mémoire
    #use_cache = False,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=50,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=True,
    fp16=True, #change to bf16=True #for XPU
    push_to_hub=True,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    save_strategy="steps",
    save_steps = 500,
    #resume_from_checkpoint=True,
    gradient_accumulation_steps=4,
    save_only_model=True,   
    #save_safetensors=True,

)



"""class CheckpointCleanupCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        clean_old_checkpoints(args.output_dir, keep=10)  # ✅ Supprime les anciens checkpoints """

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset["train"],
    eval_dataset=train_dataset["val"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early, MemoryCleanupCallback() ], #CheckpointCleanupCallback()
)
torch.cuda.empty_cache()  # Libérer la mémoire GPU inutilisée
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
trainer.save_model("/kaggle/working/best_model_final")
tokenizer.save_pretrained("/kaggle/working/best_model_final")

In [ ]:
trainer.push_to_hub()

In [ ]:
# On va maintenant evaluer notre modèle
trainer.evaluate(train_dataset['test'])

### Telechargement de mon modele du hub et enfin l'evaluer

In [ ]:
from transformers import AutoModel, AutoTokenizer
model_name = "my_train_model"  
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Modèle chargé avec succès !")


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model, 
    #args=training_args, 
    eval_dataset= train_dataset["test"],  # Utilise le dataset de test pour l'évaluation
    compute_metrics=compute_metrics 
)

results = trainer.evaluate()
print("Résultats d'évaluation :", results)


### Testons notre modele

In [ ]:
# on fait un test en traduisant une phrase
text = "comment vas-tu mon ami? J'ai pour impression que tu es triste "  
model_name = os.environ.get('HUB_ID_Fr')   
token = AutoTokenizer.from_pretrained(model_name)
inputs = token(text, return_tensors="pt", padding=True, truncation=True)
#print(token.convert_ids_to_tokens(inputs["input_ids"][0]))
# On va faire la traduction
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
outputs = model.generate(**inputs)
# On va decoder la traduction
translation = token.batch_decode(outputs, skip_special_tokens=True)
print("La traduction est :", translation)

# Le modele google T5

In [36]:
# Load model directly

tokenizer = AutoTokenizer.from_pretrained("google-t5/t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-base", token = os.environ.get('TOKEN'))

# praitretement de notre dataset et preparation pour le training
# Appliquons la tokenisation sur notre dataset
# Appliquons la tokenisation sur notre dataset
max_input_length = 128
max_target_length = 128
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["inputs"],
        max_length=max_input_length,
        truncation=True,
    )

    labels = tokenizer(
        text_target=examples["target"],
        max_length=max_target_length,
        truncation=True,
    )

    labels_ids = labels["input_ids"]

    labels_ids = [
        (l if l != tokenizer.pad_token_id else -100)
        for l in labels_ids
    ]

    model_inputs["labels"] = labels_ids
    return model_inputs
# On va appliquer la fonction sur notre dataset
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=["inputs", "target"])

# On va maintenant definir notre collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Map:   0%|          | 0/10402 [00:00<?, ? examples/s]

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1301 [00:00<?, ? examples/s]

### L'entrainement

In [40]:


training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/tmp/my_train_model_google_t5",
    run_name="nllb_experiment_v1",  # Nom de run distinct
    report_to="none",
    hub_model_id=os.environ.get('HUB_ID_Gle'),  # ✅ Ajoute ton Repo ID
    eval_strategy="steps",
    learning_rate=2e-5,
    #gradient_checkpointing=True,  # ✅ Active la réduction mémoire
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=5,
    num_train_epochs=50,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=True,
    fp16=True, #change to bf16=True for XPU
    push_to_hub=True,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    save_strategy="steps",
    gradient_accumulation_steps=4,

)




trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset["train"],
    eval_dataset=train_dataset["val"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early,MemoryCleanupCallback()],
)

trainer.train()

AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [35]:
# On va maintenant evaluer notre modèle
trainer.evaluate(train_dataset['test'])

ValueError: You should supply an encoding or a list of encodings to this method that includes input_ids, but you provided ['inputs', 'target']

## Testons le modele google

In [ ]:
# on fait un test en traduisant une phrase
text = "ammi djaye bete"
token = AutoTokenizer.from_pretrained("/kaggle/tmp/my_train_model_google_t5")
inputs = token(text, return_tensors="pt", padding=True, truncation=True)
print(token.convert_ids_to_tokens(inputs["input_ids"][0]))
# On va faire la traduction
model = AutoModelForSeq2SeqLM.from_pretrained("/kaggle/tmp/my_train_model_google_t5")
outputs = model.generate(**inputs)
# On va decoder la traduction
translation = token.batch_decode(outputs, skip_special_tokens=True)
print("La traduction est :", translation)

# Le modele bilingue Maria Henliski ang-fr

In [ ]:
# tokenisons notre ensemble de données
checkpoint = "Helsinki-NLP/opus-mt-en-fr"
# On va charger notre tokenizer et notre modèle
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint, token = os.environ.get('TOKEN'))
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Appliquons la tokenisation sur notre dataset
max_input_length = 128
max_target_length = 128
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["inputs"],
        max_length=max_input_length,
        truncation=True,
    )

    labels = tokenizer(
        text_target=examples["target"],
        max_length=max_target_length,
        truncation=True,
    )

    labels_ids = labels["input_ids"]

    labels_ids = [
        (l if l != tokenizer.pad_token_id else -100)
        for l in labels_ids
    ]

    model_inputs["labels"] = labels_ids
    return model_inputs
# On va appliquer la fonction sur notre dataset
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=["inputs", "target"])

# On va maintenant definir notre collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

In [ ]:
data_collator

In [ ]:
## Customer le Trainer pour ajouter du label_smooting 

from transformers import Seq2SeqTrainer
import torch.nn.functional as F

class CustomSeq2SeqTrainer(Seq2SeqTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]

        outputs = model(**inputs)
        logits = outputs.logits

        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
            ignore_index=-100,
            label_smoothing=0.1
        )

        return (loss, outputs) if return_outputs else loss

In [ ]:

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/my_train_henliski",
    run_name="nllb_experiment_v1",  # Nom de run distinct
    report_to="none",
    hub_model_id=os.environ.get('HUB_ID_Hs'),  # ✅ Ajoute ton Repo ID
    eval_strategy="steps",
    learning_rate=2e-5,
    #gradient_checkpointing=True,  # ✅ Active la réduction mémoire
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=50,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=True,
    fp16=True, #change to bf16=True for XPU
    push_to_hub=True,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    save_strategy="steps",
    #gradient_accumulation_steps=4,
    

)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset["train"],
    eval_dataset=train_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early,MemoryCleanupCallback()],
)

trainer.train()

In [ ]:
# On va maintenant evaluer notre modèle
trainer.evaluate()

In [ ]:
# on fait un test en traduisant une phrase
text = "ammi djaye bete"
token = AutoTokenizer.from_pretrained("/kaggle/tmp/my_train_henliski")
inputs = token(text, return_tensors="pt", padding=True, truncation=True)
print(token.convert_ids_to_tokens(inputs["input_ids"][0]))
# On va faire la traduction
model = AutoModelForSeq2SeqLM.from_pretrained("/kaggle/tmp/my_train_henliski")
outputs = model.generate(**inputs)
# On va decoder la traduction
translation = token.batch_decode(outputs, skip_special_tokens=True)
print("La traduction est :", translation)

# Plot pour les trois models 